In [1]:
import kagglehub
import pandas as pd 
import os

# Download latest version
path = kagglehub.competition_download('titanic')

print("Path to competition files:", path)

test_path = os.path.join(path, "/kaggle/input/competitions/titanic/test.csv")
train_path = os.path.join(path, "/kaggle/input/competitions/titanic/train.csv")

test_df = pd.read_csv(test_path)
train_df = pd.read_csv(train_path)

print(train_df.head())
print("="*100)
print(test_df.head())

Path to competition files: /kaggle/input/competitions/titanic
   PassengerId  Survived  Pclass  \
0            1         0       3   
1            2         1       1   
2            3         1       3   
3            4         1       1   
4            5         0       3   

                                                Name     Sex   Age  SibSp  \
0                            Braund, Mr. Owen Harris    male  22.0      1   
1  Cumings, Mrs. John Bradley (Florence Briggs Th...  female  38.0      1   
2                             Heikkinen, Miss. Laina  female  26.0      0   
3       Futrelle, Mrs. Jacques Heath (Lily May Peel)  female  35.0      1   
4                           Allen, Mr. William Henry    male  35.0      0   

   Parch            Ticket     Fare Cabin Embarked  
0      0         A/5 21171   7.2500   NaN        S  
1      0          PC 17599  71.2833   C85        C  
2      0  STON/O2. 3101282   7.9250   NaN        S  
3      0            113803  53.1000  C123     

In [2]:
print("-"*10)
print("TRAIN DATA")
print("-"*10)
print(train_df.isnull().sum())
print("-"*10)
print("TEST DATA")
print("-"*10)
print(test_df.isnull().sum())

----------
TRAIN DATA
----------
PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64
----------
TEST DATA
----------
PassengerId      0
Pclass           0
Name             0
Sex              0
Age             86
SibSp            0
Parch            0
Ticket           0
Fare             1
Cabin          327
Embarked         0
dtype: int64


In [3]:
train_df['Age'] = train_df['Age'].fillna(train_df['Age'].median())
print(train_df['Age'].isnull().sum())

0


In [4]:
train_df['Has_Cabin'] = train_df['Cabin'].apply(lambda x: 0 if pd.isna(x) else 1)
print(train_df['Cabin'].isnull().sum())
print(train_df['Has_Cabin'].isnull().sum())

687
0


In [5]:
train_df['Sex'] = train_df['Sex'].replace({'male': 0, 'female': 1})
print(train_df['Sex'])
print(train_df['Sex'].isnull().sum())

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    1
889    0
890    0
Name: Sex, Length: 891, dtype: int64
0


/tmp/ipykernel_16/3501505493.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df['Sex'] = train_df['Sex'].replace({'male': 0, 'female': 1})


In [6]:
moda_embarked = train_df['Embarked'].mode()[0]
train_df['Embarked'] = train_df['Embarked'].fillna(moda_embarked)

train_df['Embarked'] = train_df['Embarked'].replace({'C': 0, 'Q': 1, 'S': 2})
print(train_df[['Sex', 'Embarked']].head())

   Sex  Embarked
0    0         2
1    1         0
2    1         2
3    1         2
4    0         2


/tmp/ipykernel_16/185654198.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  train_df['Embarked'] = train_df['Embarked'].replace({'C': 0, 'Q': 1, 'S': 2})


In [7]:
train_df['Deck'] = train_df['Cabin'].apply(lambda x: str(x)[0] if pd.notna(x) else 'Unknown')

print(train_df.groupby('Deck')['Survived'].mean())

Deck
A          0.466667
B          0.744681
C          0.593220
D          0.757576
E          0.750000
F          0.615385
G          0.500000
T          0.000000
Unknown    0.299854
Name: Survived, dtype: float64


In [8]:
train_df = pd.get_dummies(train_df, columns=['Deck'], prefix='Deck')


deck_columns = [col for col in train_df.columns if col.startswith('Deck_')]
train_df[deck_columns] = train_df[deck_columns].astype(int)


print("Nuevas columnas de cubiertas:")
print(train_df[deck_columns].head())

Nuevas columnas de cubiertas:
   Deck_A  Deck_B  Deck_C  Deck_D  Deck_E  Deck_F  Deck_G  Deck_T  \
0       0       0       0       0       0       0       0       0   
1       0       0       1       0       0       0       0       0   
2       0       0       0       0       0       0       0       0   
3       0       0       1       0       0       0       0       0   
4       0       0       0       0       0       0       0       0   

   Deck_Unknown  
0             1  
1             0  
2             1  
3             0  
4             1  


In [9]:
train_df['FamilySize'] = train_df['SibSp'] + train_df['Parch'] + 1

print(train_df.groupby('FamilySize')['Survived'].mean())

FamilySize
1     0.303538
2     0.552795
3     0.578431
4     0.724138
5     0.200000
6     0.136364
7     0.333333
8     0.000000
11    0.000000
Name: Survived, dtype: float64


In [10]:
#Calculamos el precio por persona
train_df['Fare_PP'] = train_df['Fare'] / train_df['FamilySize']

#Agrupamos las clases sociales
train_df['Fare_Bin'] = pd.qcut(train_df['Fare_PP'], 4, labels=[0, 1, 2, 3]).astype(int)
train_df.drop(['Fare', 'Fare_PP'], axis=1, inplace=True)

In [11]:
# Extraemos el título (la palabra que termina en punto)
train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)

# Agrupamos los títulos raros o de la nobleza
train_df['Title'] = train_df['Title'].replace(['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona'], 'Rare')
train_df['Title'] = train_df['Title'].replace(['Mlle', 'Ms'], 'Miss')
train_df['Title'] = train_df['Title'].replace('Mme', 'Mrs')

# Lo convertimos a números (One-Hot Encoding)
train_df = pd.get_dummies(train_df, columns=['Title'], prefix='Title')

# Ahora sí, borramos Name y Ticket
train_df.drop(['Name', 'Ticket'], axis=1, inplace=True)

<>:2: SyntaxWarning: invalid escape sequence '\.'
<>:2: SyntaxWarning: invalid escape sequence '\.'
/tmp/ipykernel_16/279594626.py:2: SyntaxWarning: invalid escape sequence '\.'
  train_df['Title'] = train_df['Name'].str.extract(' ([A-Za-z]+)\.', expand=False)


In [12]:
# Cabin: Ya extrajimos la información útil en 'Deck' y 'Has_Cabin'.
# Sibsp y Parch: Ya tenemos la data en FamilySize
features_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'SibSp', 'Parch']

train_df.drop(columns=[col for col in features_to_drop if col in train_df.columns], inplace=True)

print("Columnas que quedaron para el modelo:")
print(train_df.columns.tolist())
print("\nVista previa de los datos listos:")
print(train_df.head())
print("\nVista de datos nulls finales")
print("-"*10)
print("TRAIN DATA")
print("-"*10)
print(train_df.isnull().sum())
print("-"*10)
print("TEST DATA")
print("-"*10)
print(test_df.isnull().sum())

Columnas que quedaron para el modelo:
['Survived', 'Pclass', 'Sex', 'Age', 'Embarked', 'Has_Cabin', 'Deck_A', 'Deck_B', 'Deck_C', 'Deck_D', 'Deck_E', 'Deck_F', 'Deck_G', 'Deck_T', 'Deck_Unknown', 'FamilySize', 'Fare_Bin', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs', 'Title_Rare']

Vista previa de los datos listos:
   Survived  Pclass  Sex   Age  Embarked  Has_Cabin  Deck_A  Deck_B  Deck_C  \
0         0       3    0  22.0         2          0       0       0       0   
1         1       1    1  38.0         0          1       0       0       1   
2         1       3    1  26.0         2          0       0       0       0   
3         1       1    1  35.0         2          1       0       0       1   
4         0       3    0  35.0         2          0       0       0       0   

   Deck_D  ...  Deck_G  Deck_T  Deck_Unknown  FamilySize  Fare_Bin  \
0       0  ...       0       0             1           2         0   
1       0  ...       0       0             0           2   

In [13]:
from sklearn.model_selection import train_test_split

X = train_df.drop('Survived', axis=1)
y = train_df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# --- Random Forest ---
grid_rf = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200, 300],
        'max_depth': [7, 8, 9, 10],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    },
    cv=5, n_jobs=-1, verbose=1
)
grid_rf.fit(X_train, y_train)
best_rf = grid_rf.best_estimator_
print(f"RF  — mejores parámetros: {grid_rf.best_params_}")

# --- Gradient Boosting ---
grid_gb = GridSearchCV(
    estimator=GradientBoostingClassifier(random_state=42),
    param_grid={
        'n_estimators': [100, 200],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 5],
        'subsample': [0.7, 0.8, 1.0]
    },
    cv=5, n_jobs=-1, verbose=1
)
grid_gb.fit(X_train, y_train)
best_gb = grid_gb.best_estimator_
print(f"GB  — mejores parámetros: {grid_gb.best_params_}")

# --- Logistic Regression ---
grid_lr = GridSearchCV(
    estimator=LogisticRegression(max_iter=1000, random_state=42),
    param_grid={
        'C': [0.01, 0.1, 0.5, 1, 10],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    },
    cv=5, n_jobs=-1, verbose=1
)
grid_lr.fit(X_train, y_train)
best_lr = grid_lr.best_estimator_
print(f"LR  — mejores parámetros: {grid_lr.best_params_}")

Fitting 5 folds for each of 216 candidates, totalling 1080 fits
RF  — mejores parámetros: {'criterion': 'entropy', 'max_depth': 9, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
Fitting 5 folds for each of 54 candidates, totalling 270 fits
GB  — mejores parámetros: {'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100, 'subsample': 0.8}
Fitting 5 folds for each of 10 candidates, totalling 50 fits
LR  — mejores parámetros: {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}


In [15]:
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import accuracy_score

ensemble = VotingClassifier(
    estimators=[('rf', best_rf), ('gb', best_gb), ('lr', best_lr)],
    voting='soft'
)
ensemble.fit(X_train, y_train)

# Comparación de los 4
models = {
    'Random Forest': best_rf,
    'Gradient Boosting': best_gb,
    'Logistic Regression': best_lr,
    'Ensemble (Voting)': ensemble
}

print("\nResultados comparativos:")
print("-" * 35)
for name, m in models.items():
    acc = accuracy_score(y_test, m.predict(X_test))
    print(f"{name:<25}: {acc:.2%}")


Resultados comparativos:
-----------------------------------
Random Forest            : 83.24%
Gradient Boosting        : 81.56%
Logistic Regression      : 82.68%
Ensemble (Voting)        : 83.80%


In [16]:
import plotly.graph_objects as go
from sklearn.metrics import roc_curve, auc

fig = go.Figure()

modelos = {
    'Random Forest': (best_rf, '#3266ad'),
    'Gradient Boosting': (best_gb, '#d85a30'),
    'Logistic Regression': (best_lr, '#639922'),
    'Ensemble (Voting)': (ensemble, '#8b5cf6')
}

for nombre, (modelo, color) in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc_score = auc(fpr, tpr)
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr, mode='lines', name=f'{nombre} (AUC={auc_score:.3f})',
        line=dict(color=color, width=2)
    ))

fig.add_trace(go.Scatter(
    x=[0,1], y=[0,1], mode='lines', name='Aleatorio',
    line=dict(color='gray', dash='dash', width=1)
))
fig.update_layout(
    title='Curva ROC — Comparación de modelos',
    xaxis_title='Tasa de Falsos Positivos (FPR)',
    yaxis_title='Tasa de Verdaderos Positivos (TPR)',
    width=750, height=480, legend=dict(x=0.55, y=0.1)
)
fig.show()

In [17]:
from sklearn.metrics import precision_recall_curve

fig = go.Figure()

for nombre, (modelo, color) in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1]
    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    best_t = thresholds[f1[:-1].argmax()]

    fig.add_trace(go.Scatter(
        x=thresholds, y=precision[:-1], mode='lines',
        name=f'{nombre} — Precisión', line=dict(color=color, width=2)
    ))
    fig.add_trace(go.Scatter(
        x=thresholds, y=recall[:-1], mode='lines',
        name=f'{nombre} — Recall', line=dict(color=color, width=2, dash='dash')
    ))
    fig.add_trace(go.Scatter(
        x=thresholds, y=f1[:-1], mode='lines',
        name=f'{nombre} — F1', line=dict(color=color, width=2, dash='dot')
    ))
    fig.add_vline(
        x=best_t, line_dash='dot', line_color=color, opacity=0.4,
        annotation_text=f'{nombre}: {best_t:.2f}',
        annotation_font=dict(color=color, size=10)
    )
    print(f"{nombre:<25} — Umbral óptimo (F1): {best_t:.3f}")

fig.update_layout(
    title='Precisión, Recall y F1 por umbral — Comparación de modelos',
    xaxis_title='Umbral', yaxis_title='Score',
    yaxis=dict(range=[0, 1.05]),
    width=750, height=500,
    legend=dict(font=dict(size=10))
)
fig.show()

Random Forest             — Umbral óptimo (F1): 0.474
Gradient Boosting         — Umbral óptimo (F1): 0.457
Logistic Regression       — Umbral óptimo (F1): 0.513
Ensemble (Voting)         — Umbral óptimo (F1): 0.555


In [18]:
fig = go.Figure()

for nombre, (modelo, color) in modelos.items():
    y_prob = modelo.predict_proba(X_test)[:, 1]
    fig.add_trace(go.Histogram(
        x=y_prob, nbinsx=30, name=nombre,
        marker_color=color, opacity=0.55
    ))

fig.add_vline(x=0.5, line_dash='dash', line_color='gray',
    annotation_text='Umbral 0.5', annotation_position='top right')
fig.update_layout(
    barmode='overlay',
    title='Distribución de probabilidades predichas — Comparación de modelos',
    xaxis_title='Probabilidad predicha de sobrevivir',
    yaxis_title='Cantidad de pasajeros',
    width=750, height=480
)
fig.show()

In [19]:
from plotly.subplots import make_subplots
from sklearn.metrics import confusion_matrix

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=list(modelos.keys()),
    horizontal_spacing=0.12, vertical_spacing=0.15
)

posiciones = [(1,1),(1,2),(2,1),(2,2)]

for (nombre, (modelo, color)), (row, col) in zip(modelos.items(), posiciones):
    cm = confusion_matrix(y_test, modelo.predict(X_test))
    fig.add_trace(go.Heatmap(
        z=cm, x=['Pred: Murió', 'Pred: Vivió'],
        y=['Real: Murió', 'Real: Vivió'],
        colorscale=[[0,'#f0f4ff'],[1, color]],
        text=cm, texttemplate='%{text}',
        showscale=False
    ), row=row, col=col)

fig.update_layout(
    title='Matrices de confusión — Comparación de modelos',
    width=750, height=600
)
fig.show()

In [20]:
import pandas as pd

fig = go.Figure()

# RF
rf_importance = pd.Series(best_rf.feature_importances_, index=X_train.columns).sort_values()
fig.add_trace(go.Bar(
    y=rf_importance.index, x=rf_importance.values,
    orientation='h', name='Random Forest',
    marker_color='#3266ad', opacity=0.8
))

# GB
gb_importance = pd.Series(best_gb.feature_importances_, index=X_train.columns).sort_values()
fig.add_trace(go.Bar(
    y=gb_importance.index, x=gb_importance.values,
    orientation='h', name='Gradient Boosting',
    marker_color='#d85a30', opacity=0.8
))

fig.update_layout(
    barmode='group',
    title='Importancia de features — RF vs Gradient Boosting',
    xaxis_title='Importancia', yaxis_title='Feature',
    width=750, height=500
)
fig.show()

In [21]:
nombres = list(modelos.keys())
accuracies = [accuracy_score(y_test, m.predict(X_test)) for _, (m, _) in modelos.items()]
colores = [c for _, (_, c) in modelos.items()]

fig = go.Figure(go.Bar(
    x=nombres, y=accuracies,
    marker_color=colores, opacity=0.85,
    text=[f'{a:.2%}' for a in accuracies],
    textposition='outside'
))
fig.update_layout(
    title='Accuracy comparativo — todos los modelos',
    yaxis=dict(range=[0.75, 0.95], title='Accuracy'),
    xaxis_title='Modelo',
    width=650, height=430
)
fig.show()